# 🖥️ Computer Vision Lab 9 — Vision Transformers vs CNNs on CODaN

**BITS F459 — Computer Vision**

Runnable companion to [`labsheet9.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet9.md).

> 📌 **This answers the question Lab 7 ended on.** A recurrent model processes a sequence
> one step at a time, so it "is slow compared with architectures that look at all positions
> in parallel". **Attention is that architecture.** Lab 8 showed what a *spatial prior*
> buys you; this lab asks what happens when you discard it — on images hard enough for the
> answer to matter.

> 📌 **Why not MNIST?** Every architecture scores ~98% on it, so the comparison says
> nothing. **CODaN** is small enough for a laptop CPU but genuinely hard, **and it ships
> separate day and night test sets** — turning Lab 8's synthetic shifted-digit experiment
> into a real-world robustness measurement.

> 📌 **Reuses Lab 8's `cnn_model.py` and `img_train.py` unchanged.** Two new files:
> `codan_dataset.py` and `vit_model.py`.

> ⚠️ **Check your kernel** (**Python (cv-dl)**) and make sure Lab 8's two files are here.

---
## 0️⃣ Setup

```bash
conda activate cv-dl
git clone https://github.com/Attila94/CODaN.git
```

The download is ~370 MB; the training split is stored as **three part-files** (GitHub caps
files at 100 MB) which must be concatenated. `codan_dataset.py` handles that — and unlike
the project's own loader it **keeps** the parts, so a failed extraction does not cost you
the whole download.

**No download or no network?** The loader falls back to a calibrated synthetic dataset, so
everything below still runs. Say which you used in your report.

| Split | Images | Source |
|---|---|---|
| `train` | 10,000 | COCO + ImageNet (daytime) |
| `val` | 500 | COCO + ImageNet (daytime) |
| `test_day` | 2,500 | COCO + ImageNet (daytime) |
| `test_night` | 2,500 | **ExDark** (genuine low-light photographs) |

**Published reference:** ResNet-18 at full 224×224 reaches **80.39% day / 48.31% night**.
You will train small models at 64×64 on a CPU, so expect far lower absolute numbers —
**judge the day→night gap, not the accuracy.**

In [ ]:
import sys, torch
print("python:", sys.executable)         # must contain 'cv-dl'
from cnn_model import SimpleCNN
from img_train import train
print("Lab 8 modules imported OK")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

---
## 1️⃣ Self-Attention from Scratch

Each token emits a **query**, a **key** and a **value**:

```
    Attention(Q, K, V) = softmax( Q·Kᵀ / √d_k ) · V
```

The weight linking two positions is **computed from their content at run time** — unlike an
MLP, where it is learned and fixed.

In [ ]:
import math
import torch


def scaled_dot_product_attention(q, k, v, scale=None):
    """q, k, v: (..., seq, d). Returns (output, attention_weights).

    attention_weights[..., i, j] = how much token i attends to token j.
    Each row sums to 1, because the softmax is over the LAST axis (j).
    """
    d_k = q.shape[-1]
    scale = scale if scale is not None else 1.0 / math.sqrt(d_k)
    scores = (q @ k.transpose(-2, -1)) * scale
    weights = scores.softmax(dim=-1)
    return weights @ v, weights

### Why `√d_k` — measure it

`q·k` sums `d_k` products, so its variance grows with dimension. Large values saturate the
softmax and almost no gradient reaches the ignored tokens.

In [ ]:
torch.manual_seed(0)
for d_k in (8, 64, 512):
    q, k = torch.randn(1, 50, d_k), torch.randn(1, 50, d_k)
    raw = q @ k.transpose(-2, -1)
    for name, s in [("unscaled", 1.0), ("scaled", 1.0 / math.sqrt(d_k))]:
        w = (raw * s).softmax(dim=-1)
        entropy = -(w * w.clamp_min(1e-9).log()).sum(-1).mean()
        print(f"d_k={d_k:4d} {name:9s} score_std={raw.std() * s:7.3f} "
              f"max_weight={w.max():.3f} entropy={entropy:.3f}")

---
## 2️⃣ `codan_dataset.py`: Real Images, Cached Small

CODaN is 224×224; training on that with a CPU is not realistic for a lab. We **resize once
and cache** — the Lab 6-2 pattern: pay once, then load a tensor instantly.

> ⚠️ **Do not hold every image in memory as a PIL object** (the official loader's approach,
> ~2.5 GB across splits). A cached `uint8` tensor is ~123 MB at 64×64 and ~31 MB at 32×32,
> and this module normalises per item rather than materialising the split as float32.

> 💡 **Resolution:** **64×64** for the worked example and Section 5; drop to **32×32** for
> the big sweeps in the exercise, or the 36-run grid becomes an overnight job.

In [ ]:
%%writefile codan_dataset.py
"""codan_dataset.py - CODaN (Common Objects Day and Night), resized and cached.

Dataset: https://github.com/Attila94/CODaN  (MIT licence)
Lengyel, Garg, Milford, van Gemert - "Zero-Shot Day-Night Domain Adaptation
with a Physics Prior", ICCV 2021.
"""

import math
import os
import shutil
import tarfile

import numpy as np
import torch
from torch.utils.data import Dataset

CODAN_CLASSES = ["Bicycle", "Car", "Motorbike", "Bus", "Boat",
                 "Cat", "Dog", "Bottle", "Cup", "Chair"]
SPLITS = ("train", "val", "test_day", "test_night")


# ------------------------------------------------------------------ unpacking
def _join_train_parts(data_dir):
    """Concatenate codan_train.tar.bz2.part{0,1,2} into one archive.

    GitHub rejects files over 100 MB, so the training split is stored in three
    pieces. Unlike the project's own loader we KEEP the parts after joining:
    deleting them means a failed extraction costs you the whole download again.
    """
    target = os.path.join(data_dir, "codan_train.tar.bz2")
    if os.path.exists(target):
        return target
    parts = [os.path.join(data_dir, f"codan_train.tar.bz2.part{i}") for i in range(3)]
    missing = [p for p in parts if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"missing train part(s): {missing}")
    print("[codan] joining 3 training parts...")
    with open(target, "wb") as out:
        for part in parts:
            with open(part, "rb") as f:
                shutil.copyfileobj(f, out)
    return target


def _extract(data_dir, split):
    out_dir = os.path.join(data_dir, split)
    if os.path.isdir(out_dir):
        return out_dir
    archive = (_join_train_parts(data_dir) if split == "train"
               else os.path.join(data_dir, f"codan_{split}.tar.bz2"))
    if not os.path.exists(archive):
        raise FileNotFoundError(archive)
    print(f"[codan] extracting {split} (one time)...")
    with tarfile.open(archive) as tar:
        try:
            # Python 3.12+ wants an explicit extraction filter; older versions
            # do not accept the argument at all.
            tar.extractall(path=data_dir, filter="data")
        except TypeError:
            tar.extractall(path=data_dir)
    return out_dir


# -------------------------------------------------------------------- caching
def _build_split(data_dir, split, size):
    """Read every JPEG of one split, resize, and stack into a uint8 tensor."""
    from PIL import Image

    split_dir = _extract(data_dir, split)
    images, targets = [], []
    for class_id, class_name in enumerate(CODAN_CLASSES):
        class_dir = os.path.join(split_dir, class_name)
        files = sorted(f for f in os.listdir(class_dir)
                       if f.lower().endswith((".jpg", ".jpeg")))
        for fname in files:
            with Image.open(os.path.join(class_dir, fname)) as im:
                im = im.convert("RGB").resize((size, size), Image.BILINEAR)
                images.append(torch.from_numpy(np.asarray(im, dtype=np.uint8)))
            targets.append(class_id)
    x = torch.stack(images).permute(0, 3, 1, 2).contiguous()   # (N, 3, size, size)
    return x, torch.tensor(targets, dtype=torch.long)


def make_synthetic_codan(split, size, seed=0, night_scale=0.53, noise=0.12):
    """Fallback: 10 synthetic classes with a built-in day/night shift.

    Each class is a hue on the colour wheel plus an oriented stripe texture,
    placed at a random position under a Gaussian envelope. Hue, orientation,
    frequency, position and brightness are all jittered per sample, so classes
    OVERLAP and the task is genuinely non-trivial rather than a giveaway.

    The 'test_night' split is darkened and noisier. `night_scale` is calibrated
    so a small CNN loses roughly 40% of its daytime accuracy at night, which is
    about what the published ResNet-18 baseline loses on real CODaN
    (80.4% -> 48.3%). Split sizes match the real dataset.

    This is a fallback, not a substitute: use it when the download is
    unavailable, and say so in your report.
    """
    counts = {"train": 10000, "val": 500, "test_day": 2500, "test_night": 2500}
    n = counts[split]
    # A FIXED offset per split: hash() is randomised per process, so using it
    # here would silently give different "synthetic CODaN" data every run.
    g = torch.Generator().manual_seed(seed + SPLITS.index(split) * 1000)
    y = torch.randint(0, len(CODAN_CLASSES), (n,), generator=g)

    coords = torch.linspace(-1, 1, size)
    gy, gx = torch.meshgrid(coords, coords, indexing="ij")
    x = torch.zeros(n, 3, size, size)
    n_cls = len(CODAN_CLASSES)
    for i in range(n):
        c = int(y[i])
        angle = c * math.pi / n_cls + 0.3 * torch.randn(1, generator=g).item()
        freq = 4 + (c % 3) + torch.randn(1, generator=g).item()
        # Class-dependent position, jittered. This matters: if the object sat
        # at a RANDOM position, class would be carried entirely by hue and
        # texture - both invariant to patch order - and Section 5.3's
        # patch-shuffle ablation would show no effect at all, because the task
        # would be a bag-of-patches problem by construction. Real objects have
        # spatial structure, so the synthetic ones must too.
        ox = 0.45 * math.cos(2 * math.pi * c / n_cls) + 0.12 * torch.randn(1, generator=g).item()
        oy = 0.45 * math.sin(2 * math.pi * c / n_cls) + 0.12 * torch.randn(1, generator=g).item()
        stripes = torch.sin(freq * ((gx - ox) * math.cos(angle)
                                    + (gy - oy) * math.sin(angle)))
        hue_angle = 2 * math.pi * c / n_cls + 0.25 * torch.randn(1, generator=g).item()
        hue = torch.tensor([0.5 + 0.5 * math.cos(hue_angle),
                            0.5 + 0.5 * math.cos(hue_angle - 2.1),
                            0.5 + 0.5 * math.cos(hue_angle + 2.1)])
        blob = torch.exp(-((gx - ox) ** 2 + (gy - oy) ** 2) / 0.6)
        x[i] = ((0.35 + 0.4 * stripes * blob)[None] * hue[:, None, None]
                + 0.15 * torch.rand(1, generator=g).item())
    x = x + noise * torch.randn(x.shape, generator=g)

    if split == "test_night":
        # A low-light shift is NOT just "darker". Fewer photons means a darker
        # image, lower contrast, a colour cast from artificial lighting, and
        # more sensor noise. Modelling only brightness would make the shift
        # trivially undoable by brightness augmentation alone, which would
        # teach the wrong lesson in Section 5.4.
        cast = torch.tensor([0.85, 0.95, 1.20])[None, :, None, None]   # cool cast
        x = x * night_scale * cast
        mu = x.mean(dim=(2, 3), keepdim=True)
        x = mu + (x - mu) * 0.75                                        # less contrast
        x = x + 0.09 * torch.randn(x.shape, generator=g)                # sensor noise

    return (x.clamp(0, 1) * 255).to(torch.uint8), y


def load_split(root="./CODaN", split="train", size=64, cache_dir="./cache"):
    """Returns (images uint8 (N,3,size,size), targets int64), cached on disk."""
    assert split in SPLITS, f"split must be one of {SPLITS}"
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"codan_{split}_{size}.pt")

    if os.path.exists(cache_path):
        blob = torch.load(cache_path)
        return blob["images"], blob["targets"]

    data_dir = os.path.join(root, "data")
    try:
        images, targets = _build_split(data_dir, split, size)
        print(f"[codan] {split}: {len(targets)} images at {size}x{size}")
    except (FileNotFoundError, OSError) as exc:
        print(f"[codan] WARNING: could not read real CODaN ({type(exc).__name__}: {exc}).")
        print("[codan] Falling back to SYNTHETIC data - say so in your report.")
        images, targets = make_synthetic_codan(split, size)

    torch.save({"images": images, "targets": targets}, cache_path)
    return images, targets


def _channel_stats(images_u8, chunk=1024):
    """Per-channel mean/std over a uint8 tensor, in chunks.

    Computed chunk-wise so we never hold the whole split as float32 at once.
    """
    total = torch.zeros(3, dtype=torch.float64)
    total_sq = torch.zeros(3, dtype=torch.float64)
    count = 0
    for i in range(0, len(images_u8), chunk):
        x = images_u8[i:i + chunk].double() / 255.0
        total += x.sum(dim=(0, 2, 3))
        total_sq += (x ** 2).sum(dim=(0, 2, 3))
        count += x.shape[0] * x.shape[2] * x.shape[3]
    mean = total / count
    var = (total_sq / count - mean ** 2).clamp_min(1e-12)
    return mean.float(), var.sqrt().float().clamp_min(1e-6)


# -------------------------------------------------------------------- dataset
class CODaNDataset(Dataset):
    """CODaN split as normalized float tensors, with optional augmentation.

    stats : (mean, std) per channel. Pass the TRAIN split's stats to every
            other split - fitting normalisation on val/test would leak, the
            same rule as Lab 4's vocabulary and Lab 6-1's StandardScaler.
    """

    def __init__(self, root="./CODaN", split="train", size=64, limit=None,
                 stats=None, augment=False, cache_dir="./cache", random_state=42):
        images, targets = load_split(root, split, size, cache_dir)

        if limit is not None and limit < len(targets):
            g = torch.Generator().manual_seed(random_state)
            keep = torch.randperm(len(targets), generator=g)[:limit]
            images, targets = images[keep], targets[keep]

        # Keep the uint8 tensor and normalise per item. Materialising the whole
        # split as float32 would cost ~500 MB at 64x64 for 10,000 images, and
        # augmentation would copy it again.
        self.images_u8 = images
        self.targets = targets
        self.augment = augment
        self.size = size

        if stats is None:                       # fit on THIS split (train only!)
            self.mean, self.std = _channel_stats(images)
        else:
            self.mean, self.std = stats

    @property
    def stats(self):
        return (self.mean, self.std)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img = self.images_u8[idx].float() / 255.0
        img = (img - self.mean[:, None, None]) / self.std[:, None, None]
        if self.augment:
            if torch.rand(1).item() < 0.5:                 # horizontal flip
                img = img.flip(-1)
            # brightness jitter: the day/night shift is largely an illumination
            # change, so this is the augmentation most likely to matter here.
            img = img * (0.6 + 0.8 * torch.rand(1).item())
        return img, self.targets[idx]


def build_codan(root="./CODaN", size=64, limit=None, augment=False, cache_dir="./cache"):
    """Returns (train, val, test_day, test_night) with TRAIN-fitted normalisation."""
    train = CODaNDataset(root, "train", size, limit, None, augment, cache_dir)
    stats = train.stats
    rest = [CODaNDataset(root, s, size, None, stats, False, cache_dir)
            for s in ("val", "test_day", "test_night")]
    return (train, *rest)

### Look at the data before modelling it

The single most useful thing you can do before training: see what the model is up against.

In [ ]:
import matplotlib.pyplot as plt
import torch

from codan_dataset import build_codan, CODAN_CLASSES

SIZE = 64
train_ds, val_ds, day_ds, night_ds = build_codan(size=SIZE)
print(f"train={len(train_ds)} val={len(val_ds)} day={len(day_ds)} night={len(night_ds)}")


def show(ds, idx, ax):
    img = ds.images_u8[idx].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.axis("off")


# Same class, day vs night - this is the domain shift, in one picture.
fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for col in range(6):
    show(day_ds, col, axes[0, col])
    axes[0, col].set_title(CODAN_CLASSES[int(day_ds.targets[col])], fontsize=9)
    show(night_ds, col, axes[1, col])
    axes[1, col].set_title(CODAN_CLASSES[int(night_ds.targets[col])], fontsize=9)
axes[0, 0].set_ylabel("day")
axes[1, 0].set_ylabel("night")
fig.suptitle("top: test_day     bottom: test_night")
plt.tight_layout()
plt.show()

# The gap measured in input space, before any model sees it.
for name, ds in [("train", train_ds), ("test_day", day_ds), ("test_night", night_ds)]:
    m = (ds.images_u8.float() / 255).mean(dim=(0, 2, 3))
    print(f"{name:11s} channel means: {[round(v, 4) for v in m.tolist()]}")

**Task.** Describe in two sentences what a model trained only on daytime images is up
against. How far apart are the train and `test_night` channel statistics — and why must
every split be normalised with the **training** statistics rather than its own?

---
## 3️⃣ `vit_model.py`

In [ ]:
%%writefile vit_model.py
"""vit_model.py - a Vision Transformer, written out block by block."""

import math

import torch
from torch import nn


def scaled_dot_product_attention(q, k, v, scale=None):
    """q, k, v: (..., seq, d) -> (output, attention weights)."""
    d_k = q.shape[-1]
    scale = scale if scale is not None else 1.0 / math.sqrt(d_k)
    scores = (q @ k.transpose(-2, -1)) * scale
    weights = scores.softmax(dim=-1)
    return weights @ v, weights


class MultiHeadSelfAttention(nn.Module):
    """One fused qkv projection, h heads, one output projection.

    Parameter layout matches nn.MultiheadAttention (in_proj_weight is [q; k; v]
    stacked along dim 0), so you can copy weights across and check - the same
    exercise as Lab 7's ManualLSTM and Lab 8's ManualConv2d.
    """

    def __init__(self, dim, num_heads=4, dropout=0.0):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError(f"dim {dim} must be divisible by num_heads {num_heads}")
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, return_attention=False):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)              # (3, B, heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        out, attn = scaled_dot_product_attention(q, k, v)
        out = out.transpose(1, 2).reshape(B, N, D)    # concatenate the heads
        out = self.drop(self.proj(out))
        return (out, attn) if return_attention else (out, None)


class TransformerBlock(nn.Module):
    """Pre-norm block: x = x + attn(norm(x)); x = x + mlp(norm(x)).

    Pre-norm rather than the original post-norm, because it trains stably
    without a learning-rate warmup - which matters when you have a lab
    session rather than a TPU pod.
    """

    def __init__(self, dim, num_heads=4, mlp_ratio=2.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout),
        )

    def forward(self, x, return_attention=False):
        h, attn = self.attn(self.norm1(x), return_attention)
        x = x + h                      # residual: the gradient highway, as in Lab 7's LSTM
        x = x + self.mlp(self.norm2(x))
        return (x, attn) if return_attention else (x, None)


class VisionTransformer(nn.Module):
    """(B, 3, img_size, img_size) -> (B, num_classes).

    pos_embed : set False to ABLATE position information (Section 5.3)
    pool      : "cls" (prepended token) | "mean" (average of patch tokens)
    Returns raw logits - softmax at display time, as in every lab since 6-1.
    """

    def __init__(self, img_size=64, patch_size=8, in_channels=3, num_classes=10,
                 dim=96, depth=4, num_heads=4, mlp_ratio=2.0, dropout=0.0,
                 pos_embed=True, pool="cls"):
        super().__init__()
        if img_size % patch_size != 0:
            raise ValueError(
                f"patch_size {patch_size} must divide img_size {img_size}; "
                f"otherwise Conv2d silently drops the leftover strip")
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.pool = pool
        self.use_pos = pos_embed

        # Patch embedding IS a stride-p convolution (Section 4).
        self.patch_embed = nn.Conv2d(in_channels, dim, patch_size, stride=patch_size)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        n_tokens = self.num_patches + (1 if pool == "cls" else 0)
        self.pos = nn.Parameter(torch.zeros(1, n_tokens, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, num_heads, mlp_ratio, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def tokenize(self, x):
        B = x.shape[0]
        t = self.patch_embed(x)                     # (B, dim, H/p, W/p)
        t = t.flatten(2).transpose(1, 2)            # (B, num_patches, dim)
        if self.pool == "cls":
            t = torch.cat([self.cls_token.expand(B, -1, -1), t], dim=1)
        if self.use_pos:
            t = t + self.pos
        return self.drop(t)

    def forward(self, x, return_attention=False):
        t = self.tokenize(x)
        attentions = []
        for block in self.blocks:
            t, attn = block(t, return_attention)
            if return_attention:
                attentions.append(attn)
        t = self.norm(t)
        summary = t[:, 0] if self.pool == "cls" else t.mean(dim=1)
        logits = self.head(summary)
        return (logits, attentions) if return_attention else logits


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Verify your attention against PyTorch

**The core deliverable of this lab** — as `ManualLSTM` was in Lab 7 and `ManualConv2d` in
Lab 8. If this passes, your attention *is* PyTorch's attention.

In [ ]:
import torch
from torch import nn
from vit_model import (VisionTransformer, MultiHeadSelfAttention,
                       scaled_dot_product_attention, count_params)

torch.manual_seed(0)
dim, heads = 96, 4
mine = MultiHeadSelfAttention(dim, heads)
ref = nn.MultiheadAttention(dim, heads, batch_first=True)
mine.qkv.weight.data.copy_(ref.in_proj_weight.data)
mine.qkv.bias.data.copy_(ref.in_proj_bias.data)
mine.proj.weight.data.copy_(ref.out_proj.weight.data)
mine.proj.bias.data.copy_(ref.out_proj.bias.data)
mine.eval(); ref.eval()

x = torch.randn(3, 65, dim)
a, _ = mine(x)
b, _ = ref(x, x, x, need_weights=False)
print("allclose:", torch.allclose(a, b, atol=1e-5), "| max diff:", (a - b).abs().max().item())

### Permutation equivariance — why positional encoding has to exist

In [ ]:
q = torch.randn(1, 10, 16)
perm = torch.randperm(10)
o1, _ = scaled_dot_product_attention(q, q, q)
o2, _ = scaled_dot_product_attention(q[:, perm], q[:, perm], q[:, perm])
print("attention is permutation-equivariant:", torch.allclose(o1[:, perm], o2, atol=1e-6))

# num_heads re-slices the SAME matrices - it adds no parameters at all.
print("params by head count:", {h: count_params(MultiHeadSelfAttention(96, h)) for h in (1, 2, 4, 8)})

# Patch embedding IS a strided convolution: Conv2d(3, dim, k=p, stride=p).
for p in (4, 8, 16):
    v = VisionTransformer(img_size=64, patch_size=p, in_channels=3)
    print(f"patch={p:2d}  tokens={tuple(v.tokenize(torch.zeros(2, 3, 64, 64)).shape)}  "
          f"patches={v.num_patches:3d}  params={count_params(v)}")

print("per-block cost:", {d: count_params(VisionTransformer(depth=d)) for d in (2, 4, 8)})

---
## 5️⃣ Worked Example: CNN vs ViT on CODaN

Lab 8's `train()` takes a ready-made `model=`, so both drop straight in — same loop, same
data, same early stopping. **That is the only reason the comparison is fair.**

In [ ]:
"""run_codan_example.py - CNN vs ViT on CODaN, day and night."""

import torch
from torch.utils.data import DataLoader
from torch import nn

from codan_dataset import build_codan, CODAN_CLASSES
from cnn_model import SimpleCNN, count_params
from img_train import train, evaluate
from vit_model import VisionTransformer

SIZE = 64
train_ds, val_ds, day_ds, night_ds = build_codan(size=SIZE, augment=False)
print(f"train={len(train_ds)} val={len(val_ds)} "
      f"day={len(day_ds)} night={len(night_ds)}")

base = dict(epochs=15, patience=4, batch_size=128, optimizer="adamw",
            lr=1e-3, weight_decay=0.05, scheduler="cosine")

criterion, device = nn.CrossEntropyLoss(), torch.device("cpu")


def both_tests(model):
    day = evaluate(model, DataLoader(day_ds, batch_size=256), criterion, device)[1]
    night = evaluate(model, DataLoader(night_ds, batch_size=256), criterion, device)[1]
    return day, night


models = {
    "cnn":  lambda: SimpleCNN(in_channels=3, num_classes=10, channels=(32, 64, 128),
                              norm="batch", input_size=SIZE),
    "vit":  lambda: VisionTransformer(img_size=SIZE, patch_size=8, in_channels=3,
                                      dim=96, depth=4, num_heads=4, dropout=0.1),
}

print(f"\n{'model':6s} {'params':>9s} {'day':>8s} {'night':>8s} {'gap':>8s} {'time_s':>8s}")
for name, factory in models.items():
    model = factory()
    res = train({**base, "run_name": f"ex_{name}", "model": name},
                train_ds, val_ds, test_ds=None, model=model)
    # train() restores the best checkpoint before returning, so this is best.pt
    day, night = both_tests(model)
    print(f"{name:6s} {res['n_params']:9d} {day:8.4f} {night:8.4f} "
          f"{day - night:8.4f} {res['train_time']:8.1f}")

### What to look for (and what *not* to expect)

- **Both models score far below the 80% ResNet-18 reference** — 12× fewer pixels, a smaller
  net, no pre-training, a CPU budget. Judge the **gap**, not the accuracy.
- **The CNN should beat the ViT**, probably by a wide margin. Correct, not a bug: 10,000
  images is nowhere near enough for a transformer to learn locality from scratch.
- **Both should lose a lot at night.** Reproducing a large gap is the headline finding.
- Note `augment=False` — the worked example shows the *raw* gap; Section 5.4 adds
  augmentation as the variable under test.

> ⚠️ **Do not tune the ViT until it wins.** The honest finding — this architecture needs
> more data than CODaN provides — *is* the lab.

### 5.3 Positional encoding — the ablation that proves the point

> ⚠️ **The order of "permute" and "add positions" is the entire experiment.** Shuffle
> *after* positions have been added and you permute whole tokens — attention is equivariant
> to that, so you measure **exactly zero change even on a model that uses positions**, and
> wrongly conclude position does not matter. The code looks right and the number is stable,
> which is what makes this trap dangerous.

In [ ]:
import torch


def shuffled_logits(model, x, perm):
    """Scramble WHICH patch sits at each position, then run the blocks.

    The order here is the whole experiment. We embed the patches, permute
    them, and only THEN add the positional embedding - so patch `perm[i]`
    arrives carrying position `i`'s embedding. The image is scrambled while
    the position information stays put.

    Doing it the other way round (calling model.tokenize(), which already
    adds positions, and permuting afterwards) permutes complete
    content+position pairs - and attention is equivariant to that, so the
    output would be unchanged and the experiment would silently prove nothing.
    """
    B = x.shape[0]
    t = model.patch_embed(x).flatten(2).transpose(1, 2)   # (B, num_patches, dim)
    t = t[:, perm]                                        # shuffle patch CONTENT
    if model.pool == "cls":
        t = torch.cat([model.cls_token.expand(B, -1, -1), t], dim=1)
    if model.use_pos:
        t = t + model.pos                                 # positions stay in place
    for block in model.blocks:
        t, _ = block(t)
    t = model.norm(t)
    summary = t[:, 0] if model.pool == "cls" else t.mean(dim=1)
    return model.head(summary)

In [ ]:
from torch.utils.data import DataLoader

from vit_model import VisionTransformer
from img_train import train


@torch.no_grad()
def accuracy(model, ds, perm=None):
    model.eval()
    correct = n = 0
    for x, y in DataLoader(ds, batch_size=256):
        out = model(x) if perm is None else shuffled_logits(model, x, perm)
        correct += (out.argmax(dim=1) == y).sum().item()
        n += len(y)
    return correct / n


n_patches = (SIZE // 8) ** 2
perm = torch.randperm(n_patches)
print(f"{'pos_embed':10s} {'day':>8s} {'shuffled':>9s} {'drop':>8s}")
for use_pos in (True, False):
    model = VisionTransformer(img_size=SIZE, patch_size=8, in_channels=3,
                              dim=96, depth=4, num_heads=4, dropout=0.1,
                              pos_embed=use_pos)
    train({**base, "run_name": f"s53_pos{use_pos}", "model": "vit"},
          train_ds, val_ds, model=model)
    a, b = accuracy(model, day_ds), accuracy(model, day_ds, perm)
    print(f"{str(use_pos):10s} {a:8.4f} {b:9.4f} {a - b:+8.4f}")

**What you should see.** The `pos_embed=False` model is **exactly** unchanged by the
shuffle — not close, identical — because it is mathematically a bag of patches. The
`pos_embed=True` model should **collapse**.

Then the harder question: without positions the model still scores well above the 10%
chance level. What can you tell about a *cat* from an unordered multiset of its 8×8
patches, and why might that work better on CODaN than on MNIST digits?

### 5.4 The day → night domain gap

**This is the experiment CODaN exists for** — the real-world counterpart of Lab 8's
shifted-MNIST test. Train on daytime images only (all `train` contains), evaluate on both.

In [ ]:
import numpy as np
from torch import nn

from cnn_model import SimpleCNN
from img_train import evaluate

criterion, device = nn.CrossEntropyLoss(), torch.device("cpu")


def day_night(model):
    d = evaluate(model, DataLoader(day_ds, batch_size=256), criterion, device)[1]
    n = evaluate(model, DataLoader(night_ds, batch_size=256), criterion, device)[1]
    return d, n


factories = {
    "cnn": lambda: SimpleCNN(in_channels=3, num_classes=10, channels=(32, 64, 128),
                             norm="batch", input_size=SIZE),
    "vit": lambda: VisionTransformer(img_size=SIZE, patch_size=8, in_channels=3,
                                     dim=96, depth=4, num_heads=4, dropout=0.1),
}

print(f"{'model':5s} {'augment':9s} {'day':>8s} {'night':>8s} {'gap':>8s} {'rel_drop':>9s}")
for aug in (False, True):
    tr_a, va_a, _, _ = build_codan(size=SIZE, augment=aug)
    for name, factory in factories.items():
        model = factory()
        train({**base, "run_name": f"s54_{name}_aug{aug}", "model": name},
              tr_a, va_a, model=model)
        d, n = day_night(model)
        print(f"{name:5s} {str(aug):9s} {d:8.4f} {n:8.4f} {d - n:8.4f} "
              f"{(d - n) / max(d, 1e-9) * 100:8.0f}%")

**Task.** Which model degrades more gracefully at night — in absolute terms, and as a
*fraction* of its own day accuracy? Does brightness augmentation close the gap, and at what
cost to day accuracy? Compare your relative drop with the published ResNet-18 figures
(80.39 → 48.31, a 40% relative drop).

### 5.5 Data efficiency — where the inductive bias earns its keep

**Use `size=32` here** to keep the runtime sane.

In [ ]:
import matplotlib.pyplot as plt

SMALL = 32
sizes = [1000, 2500, 5000, 10000]
curves = {"cnn": [], "vit": []}

_, val_s, day_s, _ = build_codan(size=SMALL)
for n in sizes:
    tr_s, _, _, _ = build_codan(size=SMALL, limit=n)
    cnn = SimpleCNN(in_channels=3, num_classes=10, channels=(32, 64, 128),
                    norm="batch", input_size=SMALL)
    train({**base, "run_name": f"eff_cnn_{n}", "model": "cnn"}, tr_s, val_s, model=cnn)
    vit = VisionTransformer(img_size=SMALL, patch_size=4, in_channels=3,
                            dim=96, depth=4, num_heads=4, dropout=0.1)
    train({**base, "run_name": f"eff_vit_{n}", "model": "vit"}, tr_s, val_s, model=vit)
    for key, m in (("cnn", cnn), ("vit", vit)):
        acc = evaluate(m, DataLoader(day_s, batch_size=256), criterion, device)[1]
        curves[key].append(acc)
    print(f"n={n:6d}  cnn={curves['cnn'][-1]:.4f}  vit={curves['vit'][-1]:.4f}  "
          f"gap={curves['cnn'][-1] - curves['vit'][-1]:+.4f}")

for name, ys in curves.items():
    plt.plot(sizes, ys, marker="o", label=name)
plt.xscale("log")
plt.xlabel("training images")
plt.ylabel("day test accuracy")
plt.title("Does the gap close as data grows?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 5.6 Attention maps

In [ ]:
import matplotlib.pyplot as plt
import torch

from codan_dataset import build_codan, CODAN_CLASSES
from vit_model import VisionTransformer

# Load a TRAINED model and real test images explicitly. Do not rely on `model`
# or `x` left over from an earlier cell - attention needs a (B, 3, H, W) image,
# and the nearest `x` in scope may well be a token tensor instead.
ckpt = torch.load("checkpoints/ex_vit/best.pt", map_location="cpu")
model = VisionTransformer(img_size=64, patch_size=8, in_channels=3,
                          dim=96, depth=4, num_heads=4, dropout=0.1)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

_, _, day_ds, _ = build_codan(size=64)
x = torch.stack([day_ds[i][0] for i in range(3)])

logits, attentions = model(x[:1], return_attention=True)
last = attentions[-1][0]                       # (heads, tokens, tokens)
cls_attn = last[:, 0, 1:]                      # what [CLS] looks at, per head
side = int(cls_attn.shape[-1] ** 0.5)

fig, axes = plt.subplots(1, cls_attn.shape[0] + 1, figsize=(13, 3))
raw = x[0].permute(1, 2, 0)
axes[0].imshow((raw - raw.min()) / (raw.max() - raw.min()))
axes[0].set_title(CODAN_CLASSES[int(day_ds[0][1])])
axes[0].axis("off")
for h, ax in enumerate(axes[1:]):
    ax.imshow(cls_attn[h].reshape(side, side).detach(), cmap="viridis")
    ax.set_title(f"head {h}")
    ax.axis("off")
plt.tight_layout()
plt.show()

> ⚠️ **Be sceptical of attention maps.** They are **not** an explanation of the
> prediction. A token can be heavily attended and barely influence the output — its value
> vector may be near zero — and residual connections carry information *around* the
> attention entirely. Repeat the plot on a **night** image: does the attention go somewhere
> sensible, or wander?

---
## 6️⃣ Inference

> ⚠️ **OpenCV reads BGR; CODaN was trained on RGB.** Forget `cvtColor` and you feed the
> network colour-swapped images — which matters far more here than on greyscale MNIST,
> because colour is a real cue for these classes.

In [ ]:
# Make a sample image so this cell runs top-to-bottom. For the exercise,
# replace my_photo.jpg with a photo you took yourself - ideally at night.
import os

import cv2

if not os.path.exists("my_photo.jpg"):
    _, _, _day, _ = build_codan(size=SIZE)
    rgb = _day.images_u8[0].permute(1, 2, 0).numpy()
    cv2.imwrite("my_photo.jpg", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    print("wrote a sample my_photo.jpg, true class =",
          CODAN_CLASSES[int(_day.targets[0])])


"""predict_codan.py - classify one image with a saved checkpoint."""

import cv2
import numpy as np
import torch

from codan_dataset import CODAN_CLASSES, build_codan
from vit_model import VisionTransformer

CHECKPOINT_PATH = "checkpoints/ex_vit/best.pt"   # hardcoded - change
IMAGE_PATH = "my_photo.jpg"
SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
print(f"Loaded epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")

model = VisionTransformer(img_size=SIZE, patch_size=8, in_channels=3,
                          dim=96, depth=4, num_heads=4).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# The normalisation MUST be the training split's statistics.
train_ds, *_ = build_codan(size=SIZE)
mean, std = train_ds.stats

bgr = cv2.imread(IMAGE_PATH)
if bgr is None:
    raise FileNotFoundError(IMAGE_PATH)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
rgb = cv2.resize(rgb, (SIZE, SIZE), interpolation=cv2.INTER_AREA)

x = torch.tensor(rgb, dtype=torch.float32).permute(2, 0, 1)[None] / 255.0
x = ((x - mean[None, :, None, None]) / std[None, :, None, None]).to(device)

with torch.no_grad():
    probs = torch.softmax(model(x), dim=1)[0]   # softmax HERE, not in forward()

pred = int(probs.argmax())
print(f"Predicted: {CODAN_CLASSES[pred]}  (confidence {probs[pred]:.3f})")
for score, idx in zip(*[t.tolist() for t in torch.topk(probs, 3)]):
    print(f"  {CODAN_CLASSES[int(idx)]:10s} {score:.3f}")

---
## 🧪 Student Exercise (Evaluative)

Full description in [`labsheet9.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet9.md).

**Part 1 — build and verify:** attention must match `nn.MultiheadAttention` to `1e-5`;
show the assert passing. Demonstrate permutation equivariance and `pos_embed=False`
shuffle-invariance. Report the dataset diagnostics.

**Part 2 — the grid (use `size=32`):** `patch_size` × `depth` × `num_heads` = 12 configs ×
3 seeds, with Lab 8's CNN at two sizes and a **parameter-matched** pair as controls.

**Part 3 — day → night:** the model × augmentation table, three seeds, plus per-class night
accuracy for your best model of each family.

**Key analysis question:** *which architecture is more robust to the night shift?* Compare
absolute and relative degradation. Does the model with **less** built-in prior transfer
better or worse — and did you expect that?

State clearly whether you used **real CODaN or the synthetic fallback**, and at what
resolution.

In [ ]:
# Starter for the parameter-matched comparison (Section 5.7).
# Tune dim/depth against channels/head until the counts are within 10%.
from cnn_model import SimpleCNN, count_params as cnn_params
from vit_model import VisionTransformer, count_params as vit_params

print("CNN options (64x64, 3 channels):")
for ch, head in [((16, 32), "gap"), ((32, 64), "gap"), ((32, 64, 128), "gap")]:
    m = SimpleCNN(in_channels=3, channels=ch, head=head, input_size=64)
    print(f"  channels={str(ch):16s} head={head:8s} {cnn_params(m):8d}")

print("ViT options (64x64, patch 8):")
for dim, depth in [(48, 4), (96, 2), (96, 4), (192, 2)]:
    m = VisionTransformer(img_size=64, patch_size=8, in_channels=3,
                          dim=dim, depth=depth, num_heads=4)
    print(f"  dim={dim:4d} depth={depth:2d}            {vit_params(m):8d}")

# TODO: pick a pair within 10%, then compare day, night, s/epoch, and limit=2500.